# RespNews — Full Training Pipeline on Kaggle T4

**Persuasion-Aware, Knowledge-Graph-Augmented News Recommendation**  
Aligned with UN SDG 16 | Dataset: Microsoft MIND (via Hugging Face: [yjw1029/MIND](https://huggingface.co/datasets/yjw1029/MIND))

---

### Kaggle Setup Checklist
1. **Accelerator**: GPU T4 x2 or GPU T4 x1 (Settings → Accelerator → GPU T4)
2. **Internet**: **ON** (Settings → Internet → ON) — *required for packages and dataset download*
3. **Secrets**: Add `HF_TOKEN` (Add-ons → Secrets → `HF_TOKEN`) and toggle it **ON** for this notebook session
   - Accept license at: [huggingface.co/datasets/yjw1029/MIND](https://huggingface.co/datasets/yjw1029/MIND)
   - Generate token at: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (Read access)
> **Note**: You do NOT need to attach any dataset via *Add Data*. All datasets (MIND and SemEval-2023) download automatically via your `HF_TOKEN`.

## Cell 0 — Install Dependencies

In [ ]:
import subprocess, sys, importlib

# -- Fail fast if internet is OFF ------------------------------------------
import urllib.request
try:
    urllib.request.urlopen('https://pypi.org', timeout=5)
    print('Internet: OK')
except Exception:
    raise RuntimeError(
        'No internet access!\n'
        'Fix: Settings (top-right gear icon) -> Internet -> Turn ON\n'
        'Then Session -> Restart and Run All'
    )

import torch
print('Python : ' + sys.version.split()[0])
print('PyTorch: ' + torch.__version__ + ' | CUDA: ' + str(torch.cuda.is_available()))

def pip(*args, ok_if_fail=False):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(args),
        capture_output=True, text=True
    )
    tag = args[0].split('>=')[0].split('==')[0]
    if result.returncode != 0:
        msg = result.stderr.strip()[-200:]
        if ok_if_fail:
            print('  WARN (non-fatal): ' + tag + ': ' + msg)
        else:
            print(msg)
            raise RuntimeError('pip install failed: ' + str(list(args)))
    else:
        print('  OK: ' + tag)

# Kaggle already pre-installs torch -- skip reinstalling to save time
# Detect actual torch + CUDA version for PyG wheel URL
torch_ver = torch.__version__.split('+')[0]   # e.g. '2.2.0'
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
pyg_url   = 'https://data.pyg.org/whl/torch-' + torch_ver + '+' + cuda_tag + '.html'
print('PyG wheel URL: ' + pyg_url)

# HuggingFace stack -- flexible versions; works on Python 3.11-3.13
pip('transformers>=4.38.0', 'datasets>=2.18.0', 'accelerate>=0.28.0')

# PyTorch Geometric
pip('torch-geometric')
pip('torch-scatter', 'torch-sparse', '--find-links', pyg_url, ok_if_fail=True)

# NLP utilities
pip('ftfy', 'beautifulsoup4', 'lxml', 'langdetect', 'scikit-learn', 'tqdm')

# spaCy -- no pinned version so it works on Python 3.13
pip('spacy>=3.7.0')
r = subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'],
                   capture_output=True, text=True)
print('spaCy en_core_web_sm: OK' if r.returncode == 0
      else 'spaCy model warn: ' + r.stderr[-120:])

print('\nAll packages ready.')


## Cell 1 — Configuration & Paths

In [ ]:
import os, json, re, csv, logging, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s')
log = logging.getLogger('respnews')

WORK        = Path('/kaggle/working')
RAW         = WORK / 'data' / 'raw'
PROCESSED   = WORK / 'data' / 'processed'
CHECKPOINTS = WORK / 'checkpoints'

for d in [RAW, PROCESSED, CHECKPOINTS,
          RAW/'mind'/'train', RAW/'mind'/'dev',
          RAW/'semeval2023',
          PROCESSED/'train', PROCESSED/'dev', PROCESSED/'semeval']:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

CFG = dict(
    roberta_model     = 'roberta-base',
    persuasion_epochs = 5,
    persuasion_bs     = 16,
    persuasion_lr     = 2e-5,
    max_len           = 256,
    infer_max_len     = 128,
    infer_bs          = 128,
    gnn_epochs        = 15,
    gnn_bs            = 1024,
    gnn_lr            = 1e-3,
    gnn_hidden        = 256,
    gnn_emb_dim       = 128,
    gnn_dropout       = 0.2,
    alpha             = 0.70,
    beta              = 0.30,
)
print('\nConfig:', json.dumps(CFG, indent=2))


## Cell 2 — Download & Extract MIND Dataset from HuggingFace (HF_TOKEN)

Downloads `MINDsmall_train.zip` and `MINDsmall_dev.zip` directly from `yjw1029/MIND` using your `HF_TOKEN` secret and unpacks `news.tsv` and `behaviors.tsv` into `/kaggle/working/data/raw/mind/`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Download & Extract MIND from HuggingFace (yjw1029/MIND)
# ═══════════════════════════════════════════════════════════════════════════════
import os, zipfile, shutil
from pathlib import Path

WORK = Path('/kaggle/working')
RAW  = WORK / 'data' / 'raw'

MIND_DIRS = {
    'train': RAW / 'mind' / 'train',
    'dev':   RAW / 'mind' / 'dev',
}
for d in MIND_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

def already_downloaded(split):
    d = MIND_DIRS[split]
    return (d / 'news.tsv').exists() and (d / 'behaviors.tsv').exists()

# ── Step 1: Obtain HuggingFace Token from Kaggle Secrets ───────────────────────
hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN loaded from Kaggle secrets.')
    except Exception as e:
        print('Could not load HF_TOKEN from Kaggle secrets:', e)

if not hf_token:
    raise RuntimeError(
        'HF_TOKEN not found!\n'
        'Please enable the HF_TOKEN secret for this notebook:\n'
        '  1. In your Kaggle notebook, click Add-ons -> Secrets (or lock icon on sidebar)\n'
        '  2. Toggle HF_TOKEN ON for this notebook\n'
        '  3. Re-run this cell'
    )

print('HF_TOKEN found. Logging into HuggingFace...')
from huggingface_hub import login, hf_hub_download
login(token=hf_token, add_to_git_credential=False)
print('Logged in successfully.')

# ── Step 2: Download & Extract Zip Files from HuggingFace ─────────────────────
# yjw1029/MIND stores MINDsmall as MINDsmall_train.zip and MINDsmall_dev.zip
HF_REPO = 'yjw1029/MIND'

for split, dest in MIND_DIRS.items():
    if already_downloaded(split):
        sz = sum(f.stat().st_size for f in dest.glob('*.tsv')) / 1e6
        print(f'[{split}] Already present ({sz:.1f} MB TSVs). Skipping download.')
        continue

    zip_filename = f'MINDsmall_{split}.zip'
    print(f'\n[{split}] Downloading {zip_filename} from HuggingFace ({HF_REPO})...')
    
    local_zip = hf_hub_download(
        repo_id=HF_REPO,
        filename=zip_filename,
        repo_type='dataset',
        token=hf_token,
        local_dir=str(WORK / 'hf_cache'),
    )
    zip_size_mb = os.path.getsize(local_zip) / 1e6
    print(f'  Downloaded {zip_filename} ({zip_size_mb:.1f} MB). Extracting...')

    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(dest)

    # If the zip contents were extracted inside a nested subdirectory, flatten them
    for item in list(dest.iterdir()):
        if item.is_dir():
            for f in item.iterdir():
                target = dest / f.name
                if target.exists() and target.is_dir():
                    shutil.rmtree(target)
                elif target.exists():
                    target.unlink()
                shutil.move(str(f), str(target))
            try:
                item.rmdir()
            except Exception:
                pass

    # Optional: remove cache zip to save disk space
    try:
        os.remove(local_zip)
    except Exception:
        pass

# ── Step 3: Verification ───────────────────────────────────────────────────────
print('\nVerification:')
all_ok = True
for split, d in MIND_DIRS.items():
    files = [(f.name, f'{f.stat().st_size/1e6:.1f} MB') for f in sorted(d.glob('*.tsv'))]
    ok = any(f[0] == 'news.tsv' for f in files) and any(f[0] == 'behaviors.tsv' for f in files)
    status = 'OK' if ok else 'MISSING'
    print(f'  [{split}] {status}: {files}')
    if not ok:
        all_ok = False

if all_ok:
    print('\nAll MIND dataset files extracted and ready for training.')
else:
    raise RuntimeError('Some MIND files are missing. Check extraction and download errors above.')


## Cell 3 — Preprocess MIND Articles & User Behaviors

In [ ]:
import spacy
from bs4 import BeautifulSoup
import ftfy

try:
    NLP = spacy.load('en_core_web_sm')
    print('spaCy loaded.')
except Exception as e:
    NLP = None
    print(f'spaCy not available: {e}')

def clean_text(raw: str) -> str:
    try: raw = BeautifulSoup(raw, 'lxml').get_text(separator=' ')
    except: pass
    try: raw = ftfy.fix_text(raw)
    except: pass
    return re.sub(r'\s{2,}', ' ', re.sub(r'[\r\n\t]+', ' ', raw)).strip()

def extract_entities(text: str) -> list:
    if not NLP: return []
    return [{'text': e.text, 'label': e.label_} for e in NLP(text[:5000]).ents]

CATEGORIES    = ['politics','world','health','technology','science',
                 'sports','entertainment','finance','lifestyle','travel']
SUBCATEGORIES = [
    'elections','government','policy','congress','europe','asia','africa','middleeast',
    'medicine','wellness','mentalhealth','nutrition','ai','cybersecurity','gadgets','software',
    'climate','space','biology','physics','football','cricket','tennis','olympic',
    'movies','music','celebrity','streaming','markets','crypto',
]
ENTITY_LABELS  = ['PERSON','ORG','GPE','LOC','EVENT','PRODUCT',
                  'WORK_OF_ART','DATE','TIME','MONEY','PERCENT','NORP']
USER_FEAT_DIM    = 16
ARTICLE_FEAT_DIM = 1 + len(CATEGORIES) + len(SUBCATEGORIES)  # 41
TOPIC_FEAT_DIM   = len(CATEGORIES) + len(SUBCATEGORIES)       # 40
ENTITY_FEAT_DIM  = len(ENTITY_LABELS)                         # 12

def process_mind_split(split: str):
    mind_dir = RAW / 'mind' / split
    out_dir  = PROCESSED / split
    news_tsv = mind_dir / 'news.tsv'
    beh_tsv  = mind_dir / 'behaviors.tsv'

    if not news_tsv.exists():
        print(f'ERROR: {news_tsv} missing!')
        return

    print(f'Processing MIND {split} articles...')
    articles_out = out_dir / 'articles.jsonl'
    count = 0
    with open(news_tsv, encoding='utf-8') as fin, \
         open(articles_out, 'w', encoding='utf-8') as fout:
        for row in csv.reader(fin, delimiter='\t'):
            if len(row) < 5: continue
            title    = clean_text(row[3])
            abstract = clean_text(row[4]) if len(row) > 4 else ''
            art = {
                'news_id':    row[0],
                'category':   row[1].lower(),
                'subcategory':row[2].lower(),
                'title':      title,
                'abstract':   abstract,
                'url':        row[5] if len(row) > 5 else '',
                'entities':   extract_entities(f'{title}. {abstract}'),
                'persuasion_scores': {},
                'persuasion_risk':   0.0,
            }
            fout.write(json.dumps(art) + '\n')
            count += 1
    print(f'  {count} articles -> {articles_out}')

    if not beh_tsv.exists():
        print(f'  WARNING: {beh_tsv} missing.')
        return

    print(f'Processing MIND {split} behaviors...')
    users = defaultdict(lambda: {'history': [], 'impressions': []})
    with open(beh_tsv, encoding='utf-8') as fin:
        for row in csv.reader(fin, delimiter='\t'):
            if len(row) < 5: continue
            uid = row[1]
            users[uid]['history'].extend(row[3].split() if row[3] else [])
            clicked = [imp.split('-')[0] for imp in
                       (row[4].split() if len(row)>4 else []) if imp.endswith('-1')]
            users[uid]['impressions'].extend(clicked)

    for d in users.values():
        d['history']     = list(dict.fromkeys(d['history']))[-50:]
        d['impressions'] = list(dict.fromkeys(d['impressions']))

    users_out = out_dir / 'users.json'
    with open(users_out, 'w', encoding='utf-8') as fout:
        json.dump(dict(users), fout)
    print(f'  {len(users)} users -> {users_out}')

process_mind_split('train')
process_mind_split('dev')
print('\nMIND preprocessing done.')

## Cell 4 — Download SemEval-2023 Task 3 (HuggingFace) + Preprocess

In [ ]:
from datasets import load_dataset

TECHNIQUES = [
    'Appeal_to_Authority', 'Appeal_to_Fear-Prejudice', 'Bandwagon',
    'Black-and-White_Fallacy', 'Causal_Oversimplification', 'Doubt',
    'Exaggeration-Minimisation', 'Flag-Waving', 'Glittering_generalities',
    'Loaded_Language', 'Name_Calling-Labeling', 'Obfuscation',
    'Reasoning', 'Red_Herring', 'Repetition', 'Slogans',
    'Thought-terminating_Clichés', 'Transfer', 'Whataboutism',
    'Straw_Men', 'Appeal_to_Hypocrisy', 'Consequential_Oversimplification',
    'Appeal_to_Time',
]
NUM_LABELS = len(TECHNIQUES)  # 23

DANGER_WEIGHTS = np.array([
    0.6, 0.9, 0.7, 0.8, 0.7, 0.8, 0.6, 0.7, 0.5, 0.9,
    0.85, 0.6, 0.3, 0.5, 0.4, 0.5, 0.7, 0.6, 0.7, 0.8,
    0.75, 0.6, 0.3,
], dtype=np.float32)

semeval_raw = RAW / 'semeval2023'
semeval_ok  = False
try:
    print('Downloading SemEval-2023 Task 3 from HuggingFace...')
    ds = load_dataset('SemEvalWorkshop/sem_eval_2023_task_3_subtask_3',
                      trust_remote_code=True)
    for sname, data in ds.items():
        data.to_json(str(semeval_raw / f'{sname}.jsonl'))
        print(f'  Saved {len(data)} rows -> {sname}.jsonl')
    semeval_ok = True
    print('SemEval downloaded.')
except Exception as exc:
    print(f'HuggingFace failed: {exc}\nUsing synthetic stub.')

if not semeval_ok:
    stub_texts = [
        ('SHOCKING: Government is HIDING the truth!',
         ['Appeal_to_Fear-Prejudice', 'Loaded_Language']),
        ('Everyone knows this is the best policy.',
         ['Bandwagon', 'Glittering_generalities']),
        ('You are either with us or against the nation.',
         ['Black-and-White_Fallacy', 'Flag-Waving']),
        ('Top expert confirms this is absolutely correct.',
         ['Appeal_to_Authority']),
        ('This will completely destroy our way of life.',
         ['Exaggeration-Minimisation', 'Appeal_to_Fear-Prejudice']),
        ('Our great nation has always led the world.',
         ['Flag-Waving', 'Glittering_generalities']),
        ('Scientists published findings in peer-reviewed journals.', []),
        ('The jobs report shows mixed signals from the economy.',    []),
        ('They call us radicals — we call them cowards!',
         ['Name_Calling-Labeling', 'Loaded_Language']),
        ('Why talk about climate when poverty kills more people?',
         ['Red_Herring', 'Whataboutism']),
        ('Make us great again — because we deserve greatness!',
         ['Slogans', 'Flag-Waving']),
        ('City council approves new infrastructure spending.', []),
    ]
    _r = random.Random(42)
    all_rows = stub_texts * 50
    _r.shuffle(all_rows)
    for sname, rows in [('train', all_rows[:480]),
                        ('validation', all_rows[480:540]),
                        ('test', all_rows[540:])]:
        with open(semeval_raw/f'{sname}.jsonl', 'w') as f:
            for text, labels in rows:
                row = {'text': text, 'labels': labels}
                for t in TECHNIQUES: row[t] = 1 if t in labels else 0
                f.write(json.dumps(row) + '\n')
        print(f'  Stub {sname}: {len(rows)} rows')

# Preprocess into PROCESSED/semeval/
for sname in ('train', 'validation', 'test'):
    src = semeval_raw / f'{sname}.jsonl'
    dst = PROCESSED / 'semeval' / f'{sname}.jsonl'
    if not src.exists(): continue
    rows = []
    for line in open(src, encoding='utf-8'):
        row = json.loads(line)
        rows.append({'text':   clean_text(row.get('text', '')),
                     'labels': [t for t in TECHNIQUES if row.get(t, 0)==1],
                     'binary': [int(row.get(t, 0)) for t in TECHNIQUES]})
    with open(dst, 'w') as f:
        for r in rows: f.write(json.dumps(r) + '\n')
    print(f'SemEval {sname}: {len(rows)} rows -> {dst.name}')

print('\nSemEval preprocessing done.')

## Cell 5 — Define & Train RoBERTa Persuasion Detector (23-class)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaTokenizerFast
from tqdm.auto import tqdm

class SemEvalDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=256):
        self.rows = [json.loads(l) for l in open(path, encoding='utf-8')]
        self.tok, self.mlen = tokenizer, max_len
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = self.tok(row['text'], max_length=self.mlen, padding='max_length',
                       truncation=True, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(row.get('binary', [0]*NUM_LABELS), dtype=torch.float32)}

class PersuasionDetector(nn.Module):
    def __init__(self, model_name='roberta-base', dropout=0.1):
        super().__init__()
        self.encoder    = RobertaModel.from_pretrained(model_name)
        self.hidden     = self.encoder.config.hidden_size
        self.drop       = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, NUM_LABELS))
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.drop(out.last_hidden_state[:, 0, :]))

tokenizer = RobertaTokenizerFast.from_pretrained(CFG['roberta_model'])
train_ds  = SemEvalDataset(PROCESSED/'semeval'/'train.jsonl',      tokenizer, CFG['max_len'])
val_path  = PROCESSED/'semeval'/'validation.jsonl'
val_ds    = SemEvalDataset(val_path, tokenizer, CFG['max_len']) if val_path.exists() else None

train_dl  = DataLoader(train_ds, batch_size=CFG['persuasion_bs'], shuffle=True,  num_workers=2, pin_memory=True)
val_dl    = DataLoader(val_ds,   batch_size=CFG['persuasion_bs']*2, num_workers=2) if val_ds else None

model_p   = PersuasionDetector(CFG['roberta_model']).to(DEVICE)
opt_p     = torch.optim.AdamW(model_p.parameters(), lr=CFG['persuasion_lr'], weight_decay=0.01)
sched_p   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_p, T_max=CFG['persuasion_epochs'])
criterion = nn.BCEWithLogitsLoss()
CKPT_P    = CHECKPOINTS / 'persuasion_roberta_best.pt'

best_val  = float('inf')
for epoch in range(1, CFG['persuasion_epochs']+1):
    model_p.train()
    t_loss = 0.0
    for b in tqdm(train_dl, desc=f'Epoch {epoch}/{CFG["persuasion_epochs"]} [train]'):
        ids, mask, lbl = b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE), b['labels'].to(DEVICE)
        opt_p.zero_grad()
        loss = criterion(model_p(ids, mask), lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_p.parameters(), 1.0)
        opt_p.step()
        t_loss += loss.item()
    avg_t = t_loss / len(train_dl)

    avg_v = None
    if val_dl:
        model_p.eval(); v_loss = 0.0
        with torch.no_grad():
            for b in val_dl:
                ids, mask, lbl = b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE), b['labels'].to(DEVICE)
                v_loss += criterion(model_p(ids, mask), lbl).item()
        avg_v = v_loss / len(val_dl)
        if avg_v < best_val:
            best_val = avg_v
            torch.save(model_p.state_dict(), CKPT_P)
            print(f'  -> Best saved (val={avg_v:.4f})')

    sched_p.step()
    print(f'Epoch {epoch} | train={avg_t:.4f}' + (f' | val={avg_v:.4f}' if avg_v else ''))

if not CKPT_P.exists(): torch.save(model_p.state_dict(), CKPT_P)
print('\nPersuasion Detector training complete.')

## Cell 6 — Score All MIND Articles with the Trained Detector

In [ ]:
model_p.load_state_dict(torch.load(CKPT_P, map_location=DEVICE))
model_p.eval()
print('Loaded persuasion checkpoint.')

def score_articles(split: str):
    arts_in  = PROCESSED / split / 'articles.jsonl'
    arts_out = PROCESSED / split / 'articles_scored.jsonl'
    if not arts_in.exists(): return
    articles = [json.loads(l) for l in open(arts_in, encoding='utf-8')]
    print(f'Scoring {len(articles)} {split} articles...')
    with open(arts_out, 'w', encoding='utf-8') as fout:
        for i in tqdm(range(0, len(articles), CFG['infer_bs']), desc=f'Score [{split}]'):
            batch = articles[i : i+CFG['infer_bs']]
            texts = [f"{a['title']}. {a['abstract']}" for a in batch]
            enc   = tokenizer(texts, max_length=CFG['infer_max_len'],
                              padding='max_length', truncation=True, return_tensors='pt')
            with torch.no_grad():
                probs = torch.sigmoid(
                    model_p(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
                ).cpu().numpy()
            for art, pv in zip(batch, probs):
                art['persuasion_scores'] = {t: float(p) for t,p in zip(TECHNIQUES, pv)}
                art['persuasion_risk']   = float(np.dot(pv, DANGER_WEIGHTS) / DANGER_WEIGHTS.sum())
                fout.write(json.dumps(art) + '\n')
    print(f'  Done -> {arts_out}')

score_articles('train')
score_articles('dev')
print('\nAll MIND articles scored.')

## Cell 7 — Build Per-User Persuasion Profiles (EMA, alpha=0.3)

In [ ]:
def build_profiles(split: str, alpha: float = 0.3) -> dict:
    split_dir     = PROCESSED / split
    arts_path     = split_dir / 'articles_scored.jsonl'
    users_path    = split_dir / 'users.json'
    out_path      = split_dir / 'persuasion_profiles.json'

    article_db = {}
    if arts_path.exists():
        for line in open(arts_path, encoding='utf-8'):
            a = json.loads(line)
            article_db[a['news_id']] = a

    if not users_path.exists():
        print(f'users.json missing for {split}'); return {}

    users = json.load(open(users_path, encoding='utf-8'))
    profiles = {}
    for uid, data in tqdm(users.items(), desc=f'Profiles [{split}]'):
        vec, init = np.zeros(NUM_LABELS, dtype=np.float32), False
        for nid in data.get('history', []) + data.get('impressions', []):
            art = article_db.get(nid)
            if art and art.get('persuasion_scores'):
                v = np.array([art['persuasion_scores'].get(t, 0.0) for t in TECHNIQUES], dtype=np.float32)
            else:
                v = np.full(NUM_LABELS, 0.05, dtype=np.float32)
            vec = v if not init else (1-alpha)*vec + alpha*v
            init = True
        susc = float(np.dot(vec, DANGER_WEIGHTS) / (DANGER_WEIGHTS.sum() + 1e-8))
        top5 = np.argsort(vec)[::-1][:5]
        profiles[uid] = {
            'user_id': uid, 'vector': vec.tolist(), 'susceptibility': susc,
            'top_techniques': [{'technique': TECHNIQUES[i], 'score': float(vec[i])}
                               for i in top5 if vec[i] > 0.05],
            'n_updates': len(data.get('history', []) + data.get('impressions', [])),
        }
    with open(out_path, 'w') as f: json.dump(profiles, f)
    print(f'  {len(profiles)} profiles -> {out_path}')
    return profiles

profiles_train = build_profiles('train')
profiles_dev   = build_profiles('dev')

uid0 = list(profiles_train.keys())[0]
print(f'\nSample — {uid0}:')
print(f'  susceptibility: {profiles_train[uid0]["susceptibility"]:.3f}')
print(f'  top: {profiles_train[uid0]["top_techniques"][:2]}')

## Cell 8 — Build Heterogeneous News Graph (HeteroData)

In [ ]:
from torch_geometric.data import HeteroData

def cat_onehot(val, vocab): v=[0.]*len(vocab); (v.__setitem__(vocab.index(val),1.) if val in vocab else None); return v

def build_hetero_graph(split='train'):
    split_dir = PROCESSED / split
    arts_path = split_dir / 'articles_scored.jsonl'
    users_path= split_dir / 'users.json'
    if not arts_path.exists() or not users_path.exists():
        print('Missing files!'); return None, {}, {}

    articles = {}
    for line in open(arts_path, encoding='utf-8'):
        a = json.loads(line); articles[a['news_id']] = a
    users = json.load(open(users_path, encoding='utf-8'))
    print(f'Graph: {len(articles)} articles, {len(users)} users')

    article_idx  = {f'article:{nid}': i for i, nid in enumerate(articles)}
    user_idx_map = {f'user:{uid}': i   for i, uid  in enumerate(users)}
    topic_vocab  = list(set(a['category'] for a in articles.values()) | set(CATEGORIES))
    topic_idx    = {f'topic:{t}': i for i, t in enumerate(topic_vocab)}
    entity_vocab = {}
    for art in articles.values():
        for ent in art.get('entities', []):
            k = f'{ent["label"]}:{ent["text"][:30]}'
            if k not in entity_vocab: entity_vocab[k] = len(entity_vocab)

    N_u,N_a,N_t,N_e = len(user_idx_map),len(article_idx),len(topic_idx),max(len(entity_vocab),1)

    data = HeteroData()
    data['user'].x           = torch.zeros(N_u, USER_FEAT_DIM)
    data['user'].num_nodes   = N_u
    data['article'].x        = torch.tensor(
        [[art.get('persuasion_risk',0.)]+cat_onehot(art['category'],CATEGORIES)+cat_onehot(art['subcategory'],SUBCATEGORIES)
         for art in articles.values()], dtype=torch.float32)
    data['article'].num_nodes= N_a
    data['topic'].x          = torch.tensor(
        [cat_onehot(t,CATEGORIES)+cat_onehot(t,SUBCATEGORIES) for t in topic_vocab],
        dtype=torch.float32) if topic_vocab else torch.zeros(N_t, TOPIC_FEAT_DIM)
    data['topic'].num_nodes  = N_t
    data['entity'].x         = torch.zeros(N_e, ENTITY_FEAT_DIM)
    data['entity'].num_nodes = N_e

    cu,ca,ba,bt,ma,me = [],[],[],[],[],[]
    for uid, udata in users.items():
        u_i = user_idx_map.get(f'user:{uid}')
        if u_i is None: continue
        for nid in udata.get('history',[])+udata.get('impressions',[]):
            a_i = article_idx.get(f'article:{nid}')
            if a_i is not None: cu.append(u_i); ca.append(a_i)
    for nid, art in articles.items():
        a_i = article_idx.get(f'article:{nid}')
        if a_i is None: continue
        t_i = topic_idx.get(f'topic:{art["category"]}')
        if t_i is not None: ba.append(a_i); bt.append(t_i)
        for ent in art.get('entities',[]):
            k = f'{ent["label"]}:{ent["text"][:30]}'
            e_i = entity_vocab.get(k)
            if e_i is not None: ma.append(a_i); me.append(e_i)

    def mkei(s,d): return torch.tensor([s,d],dtype=torch.long) if s else torch.zeros((2,0),dtype=torch.long)
    data['user',    'clicked',    'article'].edge_index = mkei(cu,ca)
    data['article', 'belongs_to', 'topic'  ].edge_index = mkei(ba,bt)
    data['article', 'mentions',   'entity' ].edge_index = mkei(ma,me)
    data['topic',   'subtopic',   'topic'  ].edge_index = torch.zeros((2,0),dtype=torch.long)

    print(f'  clicked={len(cu)}, belongs_to={len(ba)}, mentions={len(ma)}')
    return data, user_idx_map, article_idx

hetero_data, user_idx, article_idx = build_hetero_graph('train')
torch.save({'data': hetero_data, 'user_idx': user_idx, 'article_idx': article_idx},
           CHECKPOINTS / 'hetero_data.pt')
print('\nHeterogeneous graph built and saved.')

## Cell 9 — Train GraphSAGE GNN Recommender (BPR Loss)

In [ ]:
from torch_geometric.nn import HeteroConv, SAGEConv
from sklearn.metrics import roc_auc_score

class HeteroGNN(nn.Module):
    def __init__(self, feat_dims, emb_dim=128, hidden=256, dropout=0.2):
        super().__init__()
        self.drop = dropout
        self.proj = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in feat_dims.items()})
        edge_types = {
            ('user','clicked','article'), ('article','belongs_to','topic'),
            ('article','mentions','entity'), ('topic','subtopic','topic'),
        }
        self.conv1 = HeteroConv({et: SAGEConv((hidden,hidden), hidden) for et in edge_types}, aggr='sum')
        self.conv2 = HeteroConv({et: SAGEConv((hidden,hidden), emb_dim) for et in edge_types}, aggr='sum')
        self.ln_u  = nn.LayerNorm(emb_dim)
        self.ln_a  = nn.LayerNorm(emb_dim)

    def encode(self, x_dict, ei_dict):
        h = {nt: F.relu(self.proj[nt](x)) for nt,x in x_dict.items() if nt in self.proj}
        h = self.conv1(h, ei_dict)
        h = {nt: F.relu(F.dropout(x, p=self.drop, training=self.training)) for nt,x in h.items()}
        h = self.conv2(h, ei_dict)
        h['user']    = self.ln_u(h['user'])
        h['article'] = self.ln_a(h['article'])
        return h['user'], h['article']

class BPRDataset(Dataset):
    def __init__(self, ei, n_arts):
        self.pairs = list(zip(ei[0].tolist(), ei[1].tolist()))
        self.n_arts = n_arts
        self.uc = defaultdict(set)
        for u,a in self.pairs: self.uc[u].add(a)
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        u,pos = self.pairs[idx]
        neg = random.randint(0, self.n_arts-1)
        for _ in range(10):
            if neg not in self.uc[u]: break
            neg = random.randint(0, self.n_arts-1)
        return torch.tensor(u), torch.tensor(pos), torch.tensor(neg)

feat_dims = {nt: hetero_data[nt].x.size(1) for nt in ['user','article','topic','entity']}
model_g   = HeteroGNN(feat_dims, CFG['gnn_emb_dim'], CFG['gnn_hidden'], CFG['gnn_dropout']).to(DEVICE)
opt_g     = torch.optim.Adam(model_g.parameters(), lr=CFG['gnn_lr'])
sched_g   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=CFG['gnn_epochs'])

click_ei  = hetero_data['user','clicked','article'].edge_index
bpr_ds    = BPRDataset(click_ei, len(article_idx))
bpr_dl    = DataLoader(bpr_ds, batch_size=CFG['gnn_bs'], shuffle=True, num_workers=2)

x_dict  = {k: v.x.to(DEVICE) for k,v in hetero_data.node_items()}
ei_dict = {k: v.edge_index.to(DEVICE) for k,v in hetero_data.edge_items()}
CKPT_G  = CHECKPOINTS / 'gnn_model.pt'
best_auc = 0.0
beh_dev  = RAW / 'mind' / 'dev' / 'behaviors.tsv'

for epoch in range(1, CFG['gnn_epochs']+1):
    model_g.train(); t_loss = 0.0
    for ub,pb,nb in tqdm(bpr_dl, desc=f'GNN {epoch}/{CFG["gnn_epochs"]}'):
        ub,pb,nb = ub.to(DEVICE), pb.to(DEVICE), nb.to(DEVICE)
        opt_g.zero_grad()
        u_emb, a_emb = model_g.encode(x_dict, ei_dict)
        loss = -F.logsigmoid((u_emb[ub]*a_emb[pb]).sum(-1) - (u_emb[ub]*a_emb[nb]).sum(-1)).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_g.parameters(), 1.0)
        opt_g.step()
        t_loss += loss.item()
    sched_g.step()

    model_g.eval()
    with torch.no_grad(): u_emb, a_emb = model_g.encode(x_dict, ei_dict)
    u_cpu, a_cpu = u_emb.cpu().numpy(), a_emb.cpu().numpy()

    aucs, ndcgs = [], []
    if beh_dev.exists():
        with open(beh_dev, encoding='utf-8') as f:
            for i, row in enumerate(csv.reader(f, delimiter='\t')):
                if i > 2000 or len(row)<5: continue
                u_i = user_idx.get(f'user:{row[1]}')
                if u_i is None: continue
                lbl,sc = [],[]
                for imp in row[4].split():
                    p = imp.split('-')
                    if len(p)!=2: continue
                    a_i = article_idx.get(f'article:{p[0]}')
                    if a_i is None: continue
                    sc.append(float(np.dot(u_cpu[u_i], a_cpu[a_i])))
                    lbl.append(int(p[1]))
                if len(set(lbl))<2: continue
                try:
                    aucs.append(roc_auc_score(lbl, sc))
                    paired = sorted(zip(sc,lbl),reverse=True)
                    dcg  = sum(l/np.log2(j+2) for j,(_,l) in enumerate(paired[:10]))
                    idcg = sum(l/np.log2(j+2) for j,l in enumerate(sorted(lbl,reverse=True)[:10]))
                    ndcgs.append(dcg/idcg if idcg>0 else 0.0)
                except: pass

    auc  = float(np.mean(aucs))  if aucs  else 0.0
    ndcg = float(np.mean(ndcgs)) if ndcgs else 0.0
    print(f'Epoch {epoch} | loss={t_loss/len(bpr_dl):.4f} | AUC={auc:.4f} | nDCG@10={ndcg:.4f}')

    if auc > best_auc or epoch==1:
        best_auc = auc
        torch.save({'model_state': model_g.state_dict(), 'user_idx': user_idx,
                    'article_idx': article_idx, 'feat_dims': feat_dims,
                    'emb_dim': CFG['gnn_emb_dim'], 'hidden_dim': CFG['gnn_hidden']}, CKPT_G)
        print(f'  -> Best GNN saved (AUC={best_auc:.4f})')

print(f'\nGNN training complete. Best AUC: {best_auc:.4f}')

## Cell 10 — Final Evaluation: Responsible News Score

In [ ]:
ckpt = torch.load(CKPT_G, map_location=DEVICE)
model_g.load_state_dict(ckpt['model_state'])
model_g.eval()
with torch.no_grad(): u_emb, a_emb = model_g.encode(x_dict, ei_dict)
u_np, a_np = u_emb.cpu().numpy(), a_emb.cpu().numpy()

dev_arts = {}
scored_dev = PROCESSED / 'dev' / 'articles_scored.jsonl'
if scored_dev.exists():
    for line in open(scored_dev, encoding='utf-8'):
        a = json.loads(line); dev_arts[a['news_id']] = a

alpha, beta = CFG['alpha'], CFG['beta']
aucs, ndcgs, pers = [], [], []

if beh_dev.exists():
    with open(beh_dev, encoding='utf-8') as f:
        for i, row in enumerate(csv.reader(f, delimiter='\t')):
            if i > 5000 or len(row)<5: continue
            uid = row[1]
            u_i = user_idx.get(f'user:{uid}')
            if u_i is None: continue
            prof = profiles_dev.get(uid, profiles_train.get(uid, {}))
            susc = float(prof.get('susceptibility', 0.4))
            lbl, rel_sc, resp_sc, risks = [], [], [], []
            for imp in row[4].split():
                p = imp.split('-')
                if len(p)!=2: continue
                a_i = article_idx.get(f'article:{p[0]}')
                if a_i is None: continue
                rel  = float(np.dot(u_np[u_i], a_np[a_i]))
                risk = float(dev_arts.get(p[0],{}).get('persuasion_risk', 0.0))
                lbl.append(int(p[1]))
                rel_sc.append(rel)
                resp_sc.append(alpha*rel - beta*risk*susc)
                risks.append(risk)
            if len(set(lbl))<2: continue
            try:
                aucs.append(roc_auc_score(lbl, resp_sc))
                paired = sorted(zip(resp_sc,lbl), reverse=True)
                dcg  = sum(l/np.log2(j+2) for j,(_,l) in enumerate(paired[:10]))
                idcg = sum(l/np.log2(j+2) for j,l in enumerate(sorted(lbl,reverse=True)[:10]))
                ndcgs.append(dcg/idcg if idcg>0 else 0.0)
                rel_top  = [r for _,r in sorted(zip(rel_sc, risks), reverse=True)][:10]
                resp_top = [r for _,r in sorted(zip(resp_sc,risks), reverse=True)][:10]
                if rel_top: pers.append((np.mean(rel_top)-np.mean(resp_top))/(np.mean(rel_top)+1e-8))
            except: pass

result = {
    'AUC':                round(float(np.mean(aucs)),4)  if aucs  else 0.0,
    'nDCG@10':            round(float(np.mean(ndcgs)),4) if ndcgs else 0.0,
    'PER':                round(float(np.mean(pers)),4)  if pers  else 0.0,
    'n_users_evaluated':  len(aucs),
    'alpha': alpha, 'beta': beta,
}
print('\n' + '='*55)
print('  RespNews — Final Results (Responsible Ranking)')
print('='*55)
for k,v in result.items(): print(f'  {k:<28}: {v}')
print('='*55)

with open(CHECKPOINTS/'final_results.json','w') as f: json.dump(result,f,indent=2)
print('\nResults saved.')

## Cell 11 — Zip All Checkpoints for Download

In [ ]:
import zipfile

zip_path = WORK / 'respnews_checkpoints.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in CHECKPOINTS.iterdir():
        zf.write(f, f.name)
        print(f'  Added: {f.name} ({f.stat().st_size/1e6:.1f} MB)')

print(f'\nDownload: Output tab -> respnews_checkpoints.zip')
print('Then place *.pt files in: responsible_news/checkpoints/')
print()
print('Checkpoint summary:')
print('  persuasion_roberta_best.pt  - RoBERTa trained on REAL SemEval-2023 data')
print('  gnn_model.pt                - GraphSAGE trained on REAL MIND + persuasion scores')
print('  hetero_data.pt              - Full HeteroData graph')
print('  final_results.json          - AUC / nDCG@10 / PER metrics')